# Étape 8 - Le format Parquet, comparé au CSV et au JSON

1. Écrire un résultat en **Parquet**, le relire et le vérifier.
2. Écrire **le même résultat** en CSV et en JSON, puis comparer les trois formats : organisation, lisibilité, types, volume, vitesse.

- Jeu utilisé : `data/silver/sales_enriched/` (ventes enrichies, 66 593 lignes ; 39 colonnes une fois la provenance retiree).
- Sorties : `outputs/formats/` uniquement. C'est un répertoire de test dédié, qui peut être écrasé (`mode("overwrite")`).

> Les données Silver et Gold, et les sources brutes, ne sont **jamais** la cible d'une écriture dans ce notebook.

**Organisation du notebook**

| Partie | Rôle |
|---|---|
| 0. Paramètres | Chemins et session Spark lus dans `config/parametres.json` ; journal ; garde-fou sur le répertoire de sortie |
| 1 | **Écriture** Parquet dans `outputs/formats/`, **lecture** et **contrôles** demandés |
| 2 | **Écriture** du même résultat en CSV et JSON ; **contrôle** des types relus |
| 3 et 4 | Mesures de volume et de vitesse, plan d'exécution, métadonnées Parquet |
| 5 | **Résultats** : `outputs/comparaison_formats.json` |

## 0. Paramètres, chemins, session Spark et journal

Tous les paramètres et chemins viennent de `config/parametres.json` : aucun chemin n'est écrit en dur dans ce notebook.
Le journal d'exécution est ajouté dans `outputs/logs/<notebook>.log`.

In [1]:
import json
import os
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
import glob
import time

# ---------- Parametres et chemins (config/parametres.json) ----------
RACINE = os.getcwd()
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le projet")
    RACINE = os.path.dirname(RACINE)
with open(os.path.join(RACINE, "config", "parametres.json")) as f:
    PARAMS = json.load(f)


def chemin(cle):
    """Chemin absolu d'une entree de PARAMS["chemins"] (relative a la racine, ~ accepte)."""
    return os.path.normpath(os.path.join(RACINE, os.path.expanduser(PARAMS["chemins"][cle])))


LOCAL = "file://"  # sans ce prefixe, Spark irait chercher les fichiers dans HDFS (fs.defaultFS)
SOURCE = chemin("source_tpcxbb")
SILVER = LOCAL + chemin("silver")
GOLD = LOCAL + chemin("gold")
OUTPUTS = chemin("outputs")
DOSSIER_SCHEMAS = chemin("schemas")
# store_sales : echantillon ou fichier complet selon le parametre "echantillon.actif"
FICHIER_VENTES = (chemin("echantillon_ventes") if PARAMS["echantillon"]["actif"]
                  else os.path.join(SOURCE, "store_sales", "store_sales.dat"))
os.makedirs(chemin("logs"), exist_ok=True)

# ---------- Journal d'execution ----------
FICHIER_JOURNAL = os.path.join(chemin("logs"), "comparaison_formats.log")


def journal(*message):
    ligne = datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " | comparaison_formats | " + " ".join(str(m) for m in message)
    print(ligne)
    with open(FICHIER_JOURNAL, "a") as f:
        f.write(ligne + "\n")


# ---------- Session Spark (parametres "spark") ----------
constructeur = SparkSession.builder.appName("TP3-Comparaison-Formats").master(PARAMS["spark"]["master"])
for cle, valeur in PARAMS["spark"]["config"].items():
    constructeur = constructeur.config(cle, valeur)
spark = constructeur.getOrCreate()
spark.sparkContext.setLogLevel(PARAMS["spark"]["niveau_journal"])

journal("debut", "| echantillon" if PARAMS["echantillon"]["actif"] else "| fichier complet",
        "| Spark", spark.version, PARAMS["spark"]["master"])

ENTREE = os.path.join(chemin("silver"), "sales_enriched")
SORTIE = os.path.join(OUTPUTS, "formats")   # repertoire de test dedie

# Garde-fou : on n'ecrit que sous outputs/formats
assert SORTIE.startswith(OUTPUTS + os.sep)


def taille_dossier(dossier):
    """Taille des fichiers de donnees (hors fichiers techniques _SUCCESS et .crc)."""
    fichiers = [f for f in glob.glob(os.path.join(dossier, "**", "*"), recursive=True)
                if os.path.isfile(f) and not os.path.basename(f).startswith(("_", "."))]
    return sum(os.path.getsize(f) for f in fichiers), len(fichiers)


def chrono(fonction):
    debut = time.perf_counter()
    resultat = fonction()
    return resultat, round(time.perf_counter() - debut, 2)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 17:33:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-11-18 17:33:30 | comparaison_formats | debut | echantillon | Spark 4.3.0 local[2]


In [2]:
# Resultat a enregistrer : on retire les colonnes techniques de provenance,
# et on regroupe en 1 seul fichier par format pour comparer des volumes equivalents
df_resultat = spark.read.parquet(LOCAL + ENTREE).drop("_sources_silver", "_traitement", "_date_traitement")
df_resultat = df_resultat.coalesce(1).cache()
NB_LIGNES = df_resultat.count()
print("Lignes :", NB_LIGNES, "| colonnes :", len(df_resultat.columns))

Lignes : 66593 | colonnes : 39


## 1. Parquet : écrire, relire, vérifier

In [3]:
chemin_parquet = os.path.join(SORTIE, "ventes_parquet")

_, duree_ecriture_parquet = chrono(lambda: df_resultat.write.mode("overwrite").parquet(LOCAL + chemin_parquet))

df_relu = spark.read.parquet(LOCAL + chemin_parquet)
df_relu.printSchema()
df_relu.select("date_vente", "ticket_id", "nom_produit", "categorie", "nom_magasin", "quantite",
               "montant_net_paye", "marge", "taux_remise").show(10, truncate=False)
print("Nombre de lignes relues : ", df_relu.count())

root
 |-- date_vente: date (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- jour_semaine: string (nullable = true)
 |-- heure_sk: long (nullable = true)
 |-- ticket_id: long (nullable = true)
 |-- ligne_ticket: integer (nullable = true)
 |-- client_sk: long (nullable = true)
 |-- promotion_sk: long (nullable = true)
 |-- produit_sk: long (nullable = true)
 |-- produit_id: string (nullable = true)
 |-- nom_produit: string (nullable = true)
 |-- categorie_id: integer (nullable = true)
 |-- categorie: string (nullable = true)
 |-- sous_categorie: string (nullable = true)
 |-- marque: string (nullable = true)
 |-- magasin_sk: long (nullable = true)
 |-- magasin_id: string (nullable = true)
 |-- nom_magasin: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- etat: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix_catalogue_unitaire: decimal(7,2) (nullable = true)
 |-- prix_vente_unitaire: decimal(

+----------+---------+---------------------------------------------+----------------------+------------+--------+----------------+--------+-----------+
|date_vente|ticket_id|nom_produit                                  |categorie             |nom_magasin |quantite|montant_net_paye|marge   |taux_remise|
+----------+---------+---------------------------------------------+----------------------+------------+--------+----------------+--------+-----------+
|2004-01-01|53980    |Vg5Bpb290anTP6kcyUmKxyP4d6lMgoyxlgNv79M5l7   |Music                 |Williamstown|24      |516.48          |-508.56 |0.6001     |
|2004-01-01|53980    |bU9wQ9aV9KH9HIkyxJieTkrxwUuCdtrk2ZB          |Music                 |Williamstown|66      |992.64          |60.06   |0.3702     |
|2004-01-01|53980    |0I7Fj5SXVnxlzzIdb                            |Music                 |Williamstown|89      |5324.87         |-1886.80|0.2900     |
|2004-01-01|53980    |cyFFOAWYLQxSObpGhmvaedFkn315RB               |Books               

Nombre de lignes relues :  66593


### Vérifications demandées

In [4]:
verifs = {}

# 1. La lecture fonctionne
verifs["lecture"] = df_relu.count() > 0

# 2. Le schema est conserve : meme nom et meme type pour chaque colonne
verifs["schema_conserve"] = dict(df_relu.dtypes) == dict(df_resultat.dtypes)

# 3. Le nombre de lignes est coherent
verifs["nombre_lignes"] = df_relu.count() == NB_LIGNES

# 4. Les valeurs sont exploitables : on refait un calcul et on obtient le meme resultat
ca_avant = df_resultat.agg(F.sum("montant_net_paye")).first()[0]
ca_relu = df_relu.agg(F.sum("montant_net_paye")).first()[0]
verifs["valeurs_identiques"] = df_resultat.exceptAll(df_relu).count() == 0 and ca_avant == ca_relu
verifs["calcul_possible"] = ca_relu is not None

# 5. Le repertoire de sortie est correctement identifie
verifs["repertoire"] = os.path.exists(os.path.join(chemin_parquet, "_SUCCESS"))

for k, v in verifs.items():
    print(f"[{'OK' if v else 'KO'}] {k}")
print("Repertoire :", chemin_parquet.replace(RACINE + "/", ""))
print("CA relu    :", ca_relu)
assert all(verifs.values())

[OK] lecture
[OK] schema_conserve
[OK] nombre_lignes
[OK] valeurs_identiques
[OK] calcul_possible
[OK] repertoire
Repertoire : outputs/formats/ventes_parquet
CA relu    : 115752176.68


## 2. Le même résultat en CSV et en JSON

In [5]:
chemin_csv = os.path.join(SORTIE, "ventes_csv")
chemin_json = os.path.join(SORTIE, "ventes_json")

_, duree_ecriture_csv = chrono(lambda: df_resultat.write.mode("overwrite").option("header", True).csv(LOCAL + chemin_csv))
_, duree_ecriture_json = chrono(lambda: df_resultat.write.mode("overwrite").json(LOCAL + chemin_json))

# Apercu brut des fichiers : CSV et JSON sont du texte lisible, Parquet est binaire
for nom, dossier in [("CSV", chemin_csv), ("JSON", chemin_json), ("Parquet", chemin_parquet)]:
    fichier = [f for f in glob.glob(os.path.join(dossier, "part-*")) if not f.endswith(".crc")][0]
    with open(fichier, "rb") as f:
        debut = f.read(300)
    print(f"--- {nom} : {os.path.basename(fichier)}")
    print(debut.decode("utf-8", errors="replace")[:300])
    print()

--- CSV : part-00000-83263580-b538-4771-bda1-e984167f8660-c000.csv
date_vente,trimestre,mois,jour_semaine,heure_sk,ticket_id,ligne_ticket,client_sk,promotion_sk,produit_sk,produit_id,nom_produit,categorie_id,categorie,sous_categorie,marque,magasin_sk,magasin_id,nom_magasin,ville,etat,quantite,prix_catalogue_unitaire,prix_vente_unitaire,montant_catalogue,montant_rem

--- JSON : part-00000-d9c82675-58ba-4ce2-9319-db4f92ea62b0-c000.json
{"date_vente":"2004-01-01","trimestre":1,"mois":1,"jour_semaine":"Thursday","heure_sk":23617,"ticket_id":53980,"ligne_ticket":1,"client_sk":42095,"promotion_sk":61,"produit_sk":1678,"produit_id":"AAAAAAAAAAAAACMO","nom_produit":"Vg5Bpb290anTP6kcyUmKxyP4d6lMgoyxlgNv79M5l7","categorie_id":2,"categorie

--- Parquet : part-00000-8a88a786-0534-4ae4-a967-641cafc4ba77-c000.snappy.parquet
PAR1�q�q����<�  �8�?�0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0  �0 

### Gestion des types à la relecture

On relit chaque format comme le ferait un autre traitement :
- **CSV sans `inferSchema`** : c'est la lecture par défaut ;
- **CSV avec `inferSchema`** : Spark devine les types, au prix d'une passe de plus sur les données ;
- **JSON** : les types sont devinés à partir des valeurs ;
- **Parquet** : le schéma est stocké dans le fichier.

In [6]:
lectures = {
    "Parquet": spark.read.parquet(LOCAL + chemin_parquet),
    "CSV (defaut)": spark.read.option("header", True).csv(LOCAL + chemin_csv),
    "CSV (inferSchema)": spark.read.option("header", True).option("inferSchema", True).csv(LOCAL + chemin_csv),
    "JSON": spark.read.json(LOCAL + chemin_json),
}

reference = dict(df_resultat.dtypes)
COLONNES_TEMOINS = ["date_vente", "ticket_id", "quantite", "montant_net_paye", "taux_remise", "prix_nul", "annee"]

types = {nom: dict(df.dtypes) for nom, df in lectures.items()}
print(f"{'colonne':<18}{'origine':<16}" + "".join(f"{n:<20}" for n in lectures))
for c in COLONNES_TEMOINS:
    print(f"{c:<18}{reference[c]:<16}" + "".join(f"{types[n].get(c, 'absente'):<20}" for n in lectures))

ecarts_types = {nom: sum(1 for c, t in reference.items() if types[nom].get(c) != t) for nom in lectures}
print()
for nom, n in ecarts_types.items():
    print(f"{nom:<20} {n:>2} colonne(s) sur {len(reference)} n'ont plus leur type d'origine")

colonne           origine         Parquet             CSV (defaut)        CSV (inferSchema)   JSON                
date_vente        date            date                string              date                string              
ticket_id         bigint          bigint              string              int                 bigint              
quantite          int             int                 string              int                 bigint              
montant_net_paye  decimal(7,2)    decimal(7,2)        string              double              double              
taux_remise       decimal(12,4)   decimal(12,4)       string              double              double              
prix_nul          boolean         boolean             string              boolean             boolean             
annee             int             int                 string              int                 bigint              

Parquet               0 colonne(s) sur 39 n'ont plus leur type d'origine
CSV (d

In [7]:
# Consequence concrete : le CA recalcule sur chaque relecture
for nom, df in lectures.items():
    try:
        ca = df.agg(F.sum(F.col("montant_net_paye"))).first()[0]
    except Exception as e:
        ca = f"impossible : {type(e).__name__}"
    print(f"{nom:<20} CA = {ca}  (type lu : {dict(df.dtypes)['montant_net_paye']})")
print(f"{'Reference':<20} CA = {ca_avant}")

Parquet              CA = 115752176.68  (type lu : decimal(7,2))


CSV (defaut)         CA = 115752176.68000033  (type lu : string)


CSV (inferSchema)    CA = 115752176.68000033  (type lu : double)


JSON                 CA = 115752176.6800006  (type lu : double)
Reference            CA = 115752176.68


In [8]:
# Ordre des colonnes et colonnes vides : JSON ne garde pas l'ordre et omet les champs null
print("Meme ordre de colonnes que l'origine :")
for nom, df in lectures.items():
    print(f"  {nom:<20} {df.columns == df_resultat.columns}")
nb_null = df_resultat.filter(F.col("promotion_sk").isNull()).count()
print(f"\nLignes dont promotion_sk est vide : {nb_null}")
print("Exemple JSON d'une de ces lignes (le champ est absent) :")
ligne = (spark.read.text(LOCAL + chemin_json).filter(~F.col("value").contains('"promotion_sk"')).first())
print(ligne[0][:250] if ligne else "aucune")

Meme ordre de colonnes que l'origine :
  Parquet              True
  CSV (defaut)         True
  CSV (inferSchema)    True
  JSON                 False



Lignes dont promotion_sk est vide : 15
Exemple JSON d'une de ces lignes (le champ est absent) :
{"date_vente":"2004-03-01","trimestre":1,"mois":3,"jour_semaine":"Monday","heure_sk":81393,"ticket_id":56960,"ligne_ticket":1,"client_sk":51479,"produit_sk":15524,"produit_id":"AAAAAAAAAAAAAWZC","nom_produit":"SPaSflCOJcheEibTW1L12n5N2tnDfO6HW1YHBbmr


## 3. Volume de stockage observé

In [9]:
volumes = {}
for nom, dossier in [("CSV", chemin_csv), ("JSON", chemin_json), ("Parquet", chemin_parquet)]:
    octets, nb_fichiers = taille_dossier(dossier)
    volumes[nom] = octets
    print(f"{nom:<8} {octets / 1e6:>7.2f} Mo  ({nb_fichiers} fichier(s))")

for nom in ["CSV", "JSON"]:
    print(f"Parquet est {volumes[nom] / volumes['Parquet']:.1f} fois plus petit que {nom}")

CSV        28.03 Mo  (1 fichier(s))
JSON       66.18 Mo  (1 fichier(s))
Parquet     4.95 Mo  (1 fichier(s))
Parquet est 5.7 fois plus petit que CSV
Parquet est 13.4 fois plus petit que JSON


## 4. Organisation en colonnes : ce que Parquet permet

Une analyse lit souvent **quelques colonnes** parmi beaucoup d'autres : ici, 2 colonnes sur 39.
- Avec Parquet, Spark ne lit que ces colonnes (*column pruning*), et il connaît les types sans parcourir les données.
- Avec CSV et JSON, chaque ligne doit être lue en entier, puis découpée.

In [10]:
def ca_par_categorie(df):
    return (df.groupBy("categorie").agg(F.sum(F.col("montant_net_paye").cast("decimal(18,2)")).alias("ca"))
              .orderBy(F.desc("ca")).collect())

# Les schemas sont fixes a l'avance pour ne mesurer que la lecture (pas l'inference)
schema = df_resultat.schema
mesures_lecture = {}
for nom, lire in [
    ("Parquet", lambda: spark.read.parquet(LOCAL + chemin_parquet)),
    ("CSV", lambda: spark.read.schema(schema).option("header", True).csv(LOCAL + chemin_csv)),
    ("JSON", lambda: spark.read.schema(schema).json(LOCAL + chemin_json)),
]:
    ca_par_categorie(lire())  # premier passage : chauffe de la JVM
    temps = [chrono(lambda: ca_par_categorie(lire()))[1] for _ in range(3)]
    mesures_lecture[nom] = round(min(temps), 2)
    print(f"{nom:<8} CA par categorie (2 colonnes sur 39) : {mesures_lecture[nom]} s (meilleur de 3)")

Parquet  CA par categorie (2 colonnes sur 39) : 0.46 s (meilleur de 3)


CSV      CA par categorie (2 colonnes sur 39) : 0.49 s (meilleur de 3)


JSON     CA par categorie (2 colonnes sur 39) : 0.67 s (meilleur de 3)


In [11]:
# Preuve du column pruning : le plan physique Parquet ne lit que les 2 colonnes utiles
plan = (spark.read.parquet(LOCAL + chemin_parquet).groupBy("categorie").agg(F.sum("montant_net_paye"))
        ._jdf.queryExecution().executedPlan().toString())
print([l.strip() for l in plan.splitlines() if "ReadSchema" in l or "FileScan" in l][0][:400])

+- FileScan parquet [categorie#8362,montant_net_paye#8377] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/lolorigolo/DEV/J1_Retail_Intelligence/tp3_lolo/retail-intel..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<categorie:string,montant_net_paye:decimal(7,2)>


In [12]:
# Metadonnees stockees dans le pied (footer) du fichier Parquet : schema, nombre de lignes,
# compression et statistiques min/max par colonne. On les lit avec la bibliotheque Parquet de Spark.
jvm = spark._jvm
fichier = glob.glob(os.path.join(chemin_parquet, "part-*.parquet"))[0]
pied = jvm.org.apache.parquet.hadoop.ParquetFileReader.readFooter(
    spark._jsc.hadoopConfiguration(), jvm.org.apache.hadoop.fs.Path(LOCAL + fichier))
blocs = pied.getBlocks()
print("Groupes de lignes (row groups) :", blocs.size())
print("Lignes declarees dans le pied  :", sum(blocs.get(i).getRowCount() for i in range(blocs.size())))
for col in blocs.get(0).getColumns():
    nom = col.getPath().toDotString()
    if nom in ("date_vente", "categorie", "montant_net_paye"):
        print(f"{nom:<18} type {str(col.getPrimitiveType()).split(' ')[1]:<8} compression {col.getCodec()} "
              f"| {col.getStatistics()}")
print()
print(pied.getFileMetaData().getSchema().toString()[:500])

Groupes de lignes (row groups) : 1
Lignes declarees dans le pied  : 66593
date_vente         type int32    compression SNAPPY | min: 2001-01-01, max: 2005-12-13, num_nulls: 0
categorie          type binary   compression SNAPPY | min: Books, max: Toys & Games, num_nulls: 0
montant_net_paye   type int32    compression SNAPPY | min: 0.00, max: 17746.82, num_nulls: 0

message spark_schema {
  optional int32 date_vente (DATE);
  optional int32 trimestre;
  optional int32 mois;
  optional binary jour_semaine (STRING);
  optional int64 heure_sk;
  optional int64 ticket_id;
  optional int32 ligne_ticket;
  optional int64 client_sk;
  optional int64 promotion_sk;
  optional int64 produit_sk;
  optional binary produit_id (STRING);
  optional binary nom_produit (STRING);
  optional int32 categorie_id;
  optional binary categorie (STRING);
  optional binary sous_cate


## 5. Synthèse chiffrée

In [13]:
synthese = {
    "lignes": NB_LIGNES,
    "colonnes": len(df_resultat.columns),
    "volume_mo": {k: round(v / 1e6, 2) for k, v in volumes.items()},
    "ecriture_s": {"Parquet": duree_ecriture_parquet, "CSV": duree_ecriture_csv, "JSON": duree_ecriture_json},
    "lecture_ca_par_categorie_s": mesures_lecture,
    "colonnes_type_perdu": ecarts_types,
    "verifications_parquet": verifs,
}
with open(os.path.join(OUTPUTS, "comparaison_formats.json"), "w") as f:
    json.dump(synthese, f, indent=1, ensure_ascii=False)
print(json.dumps(synthese, indent=1, ensure_ascii=False))

{
 "lignes": 66593,
 "colonnes": 39,
 "volume_mo": {
  "CSV": 28.03,
  "JSON": 66.18,
  "Parquet": 4.95
 },
 "ecriture_s": {
  "Parquet": 2.67,
  "CSV": 1.41,
  "JSON": 1.1
 },
 "lecture_ca_par_categorie_s": {
  "Parquet": 0.46,
  "CSV": 0.49,
  "JSON": 0.67
 },
 "colonnes_type_perdu": {
  "Parquet": 0,
  "CSV (defaut)": 27,
  "CSV (inferSchema)": 17,
  "JSON": 18
 },
 "verifications_parquet": {
  "lecture": true,
  "schema_conserve": true,
  "nombre_lignes": true,
  "valeurs_identiques": true,
  "calcul_possible": true,
  "repertoire": true
 }
}


In [14]:
journal("fin | OK")
spark.stop()

2026-11-18 17:34:10 | comparaison_formats | fin | OK
